# Neuron Steering Lab - Colab experiment notebook

## 1. Purpose and evidence policy

Controlled study of norm-calibrated neuron-level activation steering for persona control in `Qwen/Qwen3.5-4B`.

**Evidence policy (binding):**

- `configured` != `implemented` != `tested` != `executed` != `validated`.
- Never fabricate measurements, outputs, runtimes, or agreement values.
- Mock smoke runs verify plumbing only and are never scientific evidence.
- `demo_cache/*.json` are synthetic placeholders and must never enter analysis (the runner refuses them).
- Measured report fields stay blank until accepted live records exist.
- Human-agreement fields stay blank until real annotations exist.
- Do not retune the method on evaluation results and present it as held-out evidence.


## 2. Approved storage and data handling

- Approved: this repo code, public model weights, public norm table, de-identified everyday prompts.
- NEVER upload: PHI, patient records, private conversations, credentials, restricted content, API tokens.
- Persistent outputs go under `RUNS_ROOT` (Drive optional). Colab runtimes do not survive disconnects, so every phase writes atomically and resume is supported.


In [ ]:
import os, shutil
RUNS_ROOT = "/content/runs"  # or "/content/drive/MyDrive/steering_runs" after mounting Drive
print("RUNS_ROOT:", RUNS_ROOT)
os.makedirs(RUNS_ROOT, exist_ok=True)
total, used, free = shutil.disk_usage("/content")
print(f"disk free: {free/1e9:.1f} GB (need ~25 GB for model + weights)")
bad = [p for p in ["/content/ngrok_token.txt", "/content/token.txt"] if os.path.exists(p)]
assert not bad, f"remove credential files first: {bad}"
print("storage OK; no credential files detected")


## 3. Hardware detection and resource checks

Requires a GPU runtime (T4 or better). No paid compute is started by this notebook.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>&1 | head -5
import torch
print("torch:", torch.__version__, "cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"gpu memory free/total: {free/1e9:.1f}/{total/1e9:.1f} GB")
else:
    raise SystemExit("STOP: enable a GPU runtime (Runtime > Change runtime type > T4) before any live run. Mock/tests can run without GPU.")


## 4. Verified, pinned environment setup

- Pins are explicit (no silent upgrades). Re-running install requires **Runtime > Restart runtime** before continuing.
- `transformers==4.44.2` predates Qwen3.5 support and is rejected here; this notebook configures `transformers>=4.52` and the preflight cell verifies the loading path.
- Recorded as *configured* until the preflight cell passes.


In [ ]:
# Run once, then RESTART runtime (Runtime > Restart runtime) and skip re-install.
!pip install -q "torch==2.6.0" "transformers==4.52.4" "accelerate==1.6.0" "fastapi==0.115.0" "uvicorn[standard]==0.30.6" "pydantic==2.9.2" "numpy==1.26.4" "httpx>=0.27" "pytest==8.3.3"
print("installed. NOW: Runtime > Restart runtime, then continue from section 5.")


## 5. Project and checkpoint configuration

Get the repo onto the VM (upload zip or clone - no credentials in the notebook), then freeze configuration. The 13-dose grid, 4 profiles, greedy decoding, seed 0, and max 150 tokens are preserved from the implementation.


In [ ]:
import os
PROJECT_DIR = "/content/project"  # upload this repo here, or set REPO_URL below
REPO_URL = ""  # optional public https URL; leave empty if you uploaded the repo
if REPO_URL and not os.path.exists(PROJECT_DIR):
    !git clone "$REPO_URL" "$PROJECT_DIR"
assert os.path.exists(os.path.join(PROJECT_DIR, "backend", "app.py")), f"repo not found at {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
MODEL_ID = "Qwen/Qwen3.5-4B"
DOSE_GRID = [-1.5,-1.25,-1.0,-0.75,-0.5,-0.25,0.0,0.25,0.5,0.8,1.0,1.25,1.5]
print("project:", PROJECT_DIR, "model:", MODEL_ID)


## 6. Model loading and coordinate validation

Loads the full model (slow, one time), validates every configured coordinate against the loaded layers, records the model fingerprint. Hook cleanup is verified by the unit tests with fake layers.


In [ ]:
import torch, transformers, os
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoModelForCausalLM
from backend.steering import MODEL_ID, PROFILES, get_layers
from backend.calibration import validate_coordinates, torch_model_fingerprint
from backend.runner import atomic_write_json
print("transformers:", transformers.__version__, "torch:", torch.__version__)
major, _ = torch.cuda.get_device_capability(0)
DTYPE = torch.bfloat16 if (torch.cuda.is_bf16_supported() and major >= 8) else torch.float16
print("dtype:", DTYPE, "(T4 sm_75 forces fp16)")
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
kw = dict(dtype=DTYPE, device_map="auto", trust_remote_code=True, low_cpu_mem_usage=True, attn_implementation="eager")
try:
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kw).eval()
except Exception as e:
    print("ImageTextToText failed, trying CausalLM:", e)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw).eval()
layers = get_layers(model)
print("n_layers:", len(layers))
coords = [(l, n, d) for p in PROFILES.values() for (l, n, d) in p["neurons"]]
details = validate_coordinates(layers, coords)
print("validated:", len(details), "ineligible:", [(x["layer"], x["neuron"]) for x in details if not x["eligible"]])
fp = torch_model_fingerprint(model)
print("model fingerprint:", fp["sha256"][:16], "n_params:", fp["n_params"])
atomic_write_json(os.path.join(RUNS_ROOT, "preflight.json"),
    {"model_id": MODEL_ID, "torch": torch.__version__, "transformers": transformers.__version__,
     "dtype": str(DTYPE), "gpu": torch.cuda.get_device_name(0), "n_layers": len(layers),
     "model_fingerprint": fp, "coordinates": details})
print("wrote " + RUNS_ROOT + "/preflight.json")


## 7. Calibration verification

Accepted runs require verified medians. L5/L15 ship no stored median, so they resolve to the unverified `10.0` fallback and are flagged **ineligible** until remeasured under an approved protocol.


In [ ]:
from backend.calibration import median_with_source, calibration_fingerprint, get_median_strict
from backend.steering import PROFILES
print("calibration fingerprint:", calibration_fingerprint()["sha256"][:16])
for name, p in PROFILES.items():
    for (l, n, d) in p["neurons"]:
        v, s = median_with_source(l)
        print(f"{name:10s} L{l} N{n}: median={v} source={s} eligible={s != 'fallback'}")
try:
    get_median_strict(15)
    print("strict L15: verified")
except Exception as e:
    print("strict L15 correctly refused:", str(e)[:120])


## 8. Unit tests

CPU-only; no model needed. Must pass before any live run.


In [ ]:
!cd "$PROJECT_DIR" && python -m pytest backend/tests -q 2>&1 | tail -3


## 9. Small end-to-end smoke run (mock plumbing - NOT evidence)

Uses the mock backend (no model). Separate run ID. Verifies manifest states, atomic writes, resume, and synthetic-data rejection.


In [ ]:
import os
os.environ["MOCK_MODE"] = "1"
from fastapi.testclient import TestClient
from backend.app import app
from backend.runner import default_config, init_run, set_state, run_sweep_phase, aggregate_sweep
cfg = default_config(doses=[0.0, 0.8], max_tokens=20, backend="mock")
run_dir, m = init_run(RUNS_ROOT, cfg, ["sweep"])
set_state(run_dir, "preflight_passed")
print("run:", run_dir)
print(run_sweep_phase(TestClient(app), run_dir, cfg))
set_state(run_dir, "completed", "sweep", "completed")
try:
    aggregate_sweep(run_dir, only_live=True)
except Exception as e:
    print("live aggregation correctly refused mock rows:", str(e)[:150])
print(aggregate_sweep(run_dir, only_live=False))


## 10. Small live smoke run (loads the FULL model)

Guarded: set `RUN_LIVE_SMOKE = True` only after sections 6-8 pass. One prompt x 2 doses x short generations. Still plumbing, not scientific completion.


In [ ]:
RUN_LIVE_SMOKE = False  # set True to run (loads full model, GPU minutes)
if not RUN_LIVE_SMOKE:
    print("skipped: set RUN_LIVE_SMOKE=True after preflight passes")
else:
    import os
    os.environ["MOCK_MODE"] = "0"
    from fastapi.testclient import TestClient
    from backend.app import app, state
    import backend.app as appmod
    from backend.runner import default_config, init_run, set_state, run_sweep_phase
    appmod.load_model()
    assert state["model"] is not None, f"model failed: {state['load_error']}"
    cfg = default_config(doses=[0.0, 0.8], max_tokens=30, backend="live")
    run_dir, m = init_run(RUNS_ROOT, cfg, ["sweep"])
    print("live smoke run:", run_dir)
    print(run_sweep_phase(TestClient(app), run_dir, cfg))
    print("manifest: " + run_dir + "/manifest.json")


## 11. Resource estimate from observed smoke timings (estimates, not measurements)

Extrapolates from a *completed live smoke manifest only*. If none exists, no estimate is given.


In [ ]:
import glob, json, os
cands = sorted(glob.glob(os.path.join(RUNS_ROOT, "*/manifest.json")))
live = [p for p in cands if json.load(open(p)).get("config", {}).get("backend") == "live"]
if not live:
    print("no completed live smoke manifest: no estimate available (run section 10 first)")
else:
    m = json.load(open(live[-1]))
    s = m["timing_s"].get("sweep", 0); calls = max(1, m["counts"].get("model_calls", 1))
    per_call = s / calls
    est_h = per_call * (100 * 4 + 20 * 4 + 50) / 3600
    print(f"ESTIMATE from {m['run_id']}: {per_call:.1f}s per sweep-call -> full-study upper bound ~{est_h:.1f} GPU-h (estimate, excludes judging)")


## 12. Full experiment selection (approval gate)

The full automated study (dose/specificity/sign/composition/ablation/thresholds/lens + judging) is **pending**: no 100-prompt evaluation set, no judge, and no GPU execution exist in this workspace. The runner supports resumable sweep batches; extend `PHASES` only after the section-11 estimate is reviewed and compute is approved.

To run anything expensive, set BOTH flags below to True after approval. Nothing runs on import or on opening this notebook.


In [ ]:
RUN_FULL = False  # approval gate 1
I_APPROVE_EXPENSIVE_RUN = False  # approval gate 2 (set True only with compute approval)
PHASES = ["sweep"]  # implemented phase; others pending: dose,specificity,sign,composition,ablation,thresholds,lens
print("gates:", RUN_FULL, I_APPROVE_EXPENSIVE_RUN, "phases:", PHASES)


## 13. Full run (guarded)


In [ ]:
if not (RUN_FULL and I_APPROVE_EXPENSIVE_RUN):
    print("blocked: set RUN_FULL=True and I_APPROVE_EXPENSIVE_RUN=True after approval (section 12)")
else:
    import os
    os.environ["MOCK_MODE"] = "0"
    from backend.runner import default_config, init_run, set_state, run_sweep_phase
    from backend.app import app
    from fastapi.testclient import TestClient
    raise SystemExit("full 100-prompt study not implemented in this workspace: extend runner phases first")


## 14. Progress and checkpoint status


In [ ]:
import glob, json, os
for p in sorted(glob.glob(os.path.join(RUNS_ROOT, "*/manifest.json"))):
    m = json.load(open(p))
    print(m["run_id"], m["state"], m["phases_status"], m["counts"], m.get("artifacts", []))


## 15. Resume an interrupted run

Re-running the phase with the same run directory and identical config reuses matching records (fingerprint-checked) instead of regenerating them.


In [ ]:
RESUME_RUN_DIR = ""  # paste a run dir from section 14, e.g. RUNS_ROOT + "/smoke-..."
if not RESUME_RUN_DIR:
    print("set RESUME_RUN_DIR to resume")
else:
    import os
    os.environ["MOCK_MODE"] = "1"
    from fastapi.testclient import TestClient
    from backend.app import app
    from backend.runner import load_manifest, run_sweep_phase, default_config
    m = load_manifest(RESUME_RUN_DIR)
    print(run_sweep_phase(TestClient(app), RESUME_RUN_DIR, m["config"]))


## 16. Aggregate and validate results (live only)

Accepts only records with `mock:false, label:live`. Mock rows raise instead of entering analysis.


In [ ]:
AGG_RUN_DIR = ""  # paste run dir to aggregate
if not AGG_RUN_DIR:
    print("set AGG_RUN_DIR to aggregate")
else:
    from backend.runner import aggregate_sweep, load_manifest
    agg = aggregate_sweep(AGG_RUN_DIR, only_live=True)
    print("aggregated", len(agg["per_dose_keyword"]), "doses;", "wrote", AGG_RUN_DIR + "/aggregates.json")
    m = load_manifest(AGG_RUN_DIR)
    print("state:", m["state"], "counts:", m["counts"])


## 17. Plots, results.txt, annotation export, and bundle export

- Plots are generated ONLY from accepted live aggregates; with no live records there are no plots (missing stays missing).
- `results.txt` is assembled from run manifests (never hand-typed numbers).
- Annotation export is blinded; agreement stays blank until real labels arrive.


In [ ]:
import glob, json, os
from backend.runner import atomic_write_json
lines = ["NEURON STEERING LAB - results (assembled from manifests, not hand-typed)", ""]
n_live = 0
for p in sorted(glob.glob(os.path.join(RUNS_ROOT, "*/manifest.json"))):
    m = json.load(open(p))
    live = m.get("config", {}).get("backend") == "live"
    n_live += live
    lines.append(f"{m['run_id']}: state={m['state']} backend={m['config'].get('backend')} counts={m['counts']}")
lines += ["", f"live runs present: {n_live}",
          "measured dose/persona/judge/threshold tables: PENDING (no accepted live records)",
          "plots: none (no accepted live data; will be generated from aggregates.json only)",
          "human agreement: PENDING (no annotations)"]
open(os.path.join(RUNS_ROOT, "results.txt"), "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
print("annotation: use backend.annotate.export_items on accepted live records, then validate_labels on import; agreement=None until real labels")
!tar -czf /content/steering_bundle.tgz -C "$RUNS_ROOT" . 2>&1 | tail -1; ls -la /content/steering_bundle.tgz
